# Local Whisper no Google Colab

Gera legendas `.srt` em pt-BR a partir de vídeos no Google Drive usando [faster-whisper](https://github.com/SYSTRAN/faster-whisper) com GPU e inferência batched.

**Antes de rodar:** `Ambiente de execução` → `Alterar tipo de ambiente de execução` → **T4 GPU**.

Fluxo:
1. Verificar GPU
2. Instalar dependências
3. Montar o Google Drive
4. Configurar pasta de entrada, modelo e `BATCH_SIZE`
5. Carregar o modelo via `BatchedInferencePipeline`
6. Transcrever e salvar `.srt` ao lado de cada vídeo

> O `BatchedInferencePipeline` processa várias janelas de VAD em paralelo na mesma GPU — ganho típico de **2–4×** sobre o modo sequencial. Em troca, `condition_on_previous_text` é forçado a `False` (os batches são independentes), o que raramente é perceptível na saída.

## 1. Verificar GPU

Se `nvidia-smi` falhar, o ambiente não está com GPU — troque o runtime antes de continuar.

In [ ]:
!nvidia-smi

## 2. Instalar dependências

`av<14` reproduz a pin do projeto local. O `ctranslate2` precisa de CUDA 12 e cuDNN 9, que o runtime do Colab pode não trazer. Por isso a célula instala essas bibliotecas via pip e as registra no sistema com `ldconfig`.

In [ ]:
!pip install -q "faster-whisper>=1.2.1" "av<14" nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"

import site
import subprocess
from pathlib import Path

# LD_LIBRARY_PATH só vale no início do processo; o ldconfig funciona com o kernel já rodando.
cuda_lib_dirs = set()
for packages_dir in site.getsitepackages():
    for lib_name in ("libcublas.so.12", "libcudnn.so.9"):
        for lib in Path(packages_dir).glob(f"nvidia/**/{lib_name}"):
            cuda_lib_dirs.add(str(lib.parent))

if not cuda_lib_dirs:
    raise RuntimeError("libcublas.so.12 e libcudnn.so.9 não encontradas após o pip install")

Path("/etc/ld.so.conf.d/cuda12-pip.conf").write_text("\n".join(sorted(cuda_lib_dirs)) + "\n")
subprocess.run(["ldconfig"], check=True)

ldconfig_output = subprocess.run(["ldconfig", "-p"], capture_output=True, text=True).stdout
for lib_name in ("libcublas.so.12", "libcudnn.so.9"):
    status = "ok" if lib_name in ldconfig_output else "NÃO ENCONTRADA"
    print(f"[*] {lib_name}: {status}")

## 3. Montar o Google Drive

Os vídeos devem estar em alguma pasta do seu Drive. Cada legenda será salva com o mesmo nome do vídeo (`video.mp4` → `video.srt`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Configuração

- **INPUT_PATH**: pasta ou arquivo de vídeo dentro do Drive.
- **SCAN_DEPTH**: quantos níveis de subpastas varrer abaixo de `INPUT_PATH`. `0` = só a própria pasta; `1` = filmes (`filmes/Nome do Filme/filme.mkv`); `2` = séries (`series/Nome da Série/Season 01/ep.mkv`).
- **OUTPUT_PATH**: pasta de saída, que espelha as subpastas de `INPUT_PATH` (vazio = salvar ao lado do vídeo).
- **MODEL_SIZE**: `large-v3` aproveita o T4 e dá o melhor resultado; use `small` para igualar o setup local.
- **BATCH_SIZE**: janelas de VAD processadas em paralelo na GPU. `8`–`16` costuma ser o sweet spot no T4 com `large-v3`; baixe se estourar VRAM, suba em GPUs maiores.
- **MAX_SUBTITLE_SECONDS**: duração máxima de cada legenda.
- **MAX_SUBTITLE_CHARS**: tamanho máximo de cada legenda, em caracteres. `84` equivale a duas linhas de 42.
- **MAX_GAP_SECONDS**: pausa entre palavras que força uma nova legenda.
- **FORCE**: `True` reprocessa mesmo que já exista uma legenda.

In [ ]:
INPUT_PATH = "/content/drive/MyDrive/videos"  # @param {type:"string"}
SCAN_DEPTH = 2  # @param {type:"integer"}
OUTPUT_PATH = ""  # @param {type:"string"}
MODEL_SIZE = "large-v3"  # @param ["tiny", "base", "small", "medium", "large-v3"]
LANGUAGE = "pt"  # @param {type:"string"}
BEAM_SIZE = 5  # @param {type:"integer"}
BATCH_SIZE = 8  # @param {type:"integer"}
MAX_SUBTITLE_SECONDS = 6.0  # @param {type:"number"}
MAX_SUBTITLE_CHARS = 84  # @param {type:"integer"}
MAX_GAP_SECONDS = 1.0  # @param {type:"number"}
FORCE = False  # @param {type:"boolean"}

## 5. Funções auxiliares

Adaptação do pacote `local_whisper` (`files.py`, `srt.py`, `transcriber.py`) para rodar em um único notebook.

In [ ]:
from __future__ import annotations

import os
import time
from pathlib import Path
from typing import NamedTuple

from tqdm.auto import tqdm

VIDEO_EXTENSIONS = {".mp4", ".mkv", ".avi", ".mov", ".webm", ".flv", ".wmv", ".m4v", ".ts"}

SUBTITLE_EXTENSIONS = {
    ".srt", ".vtt", ".ass", ".ssa", ".sub", ".sbv",
    ".ttml", ".dfxp", ".smi", ".stl", ".usf", ".idx",
}

SENTENCE_ENDINGS = (".", "!", "?", "…")


class Cue(NamedTuple):
    start: float
    end: float
    text: str


def collect_videos(path: str, depth: int) -> list[Path]:
    root_path = Path(path)
    if root_path.is_file():
        return [root_path] if root_path.suffix.lower() in VIDEO_EXTENSIONS else []

    videos: list[Path] = []
    for current_dir, subdirs, filenames in os.walk(root_path):
        current_path = Path(current_dir)
        level = len(current_path.relative_to(root_path).parts)
        if level >= depth:
            subdirs.clear()
        for filename in filenames:
            file_path = current_path / filename
            if file_path.suffix.lower() in VIDEO_EXTENSIONS:
                videos.append(file_path)
    return sorted(videos)


def existing_subtitle(video: Path, extra_dir: Path | None = None) -> Path | None:
    dirs = [video.parent]
    if extra_dir and extra_dir != video.parent:
        dirs.append(extra_dir)
    for d in dirs:
        for ext in SUBTITLE_EXTENSIONS:
            candidate = d / (video.stem + ext)
            if candidate.exists():
                return candidate
    return None


def words_to_text(words: list) -> str:
    return "".join(word.word for word in words).strip()


def should_start_new_cue(current: list, word, max_seconds: float,
                         max_chars: int, max_gap: float) -> bool:
    previous = current[-1]
    ends_sentence = previous.word.strip().endswith(SENTENCE_ENDINGS)
    has_long_gap = word.start - previous.end > max_gap
    too_long = word.end - current[0].start > max_seconds
    too_many_chars = len(words_to_text(current + [word])) > max_chars
    return ends_sentence or has_long_gap or too_long or too_many_chars


def segments_to_cues(segments: list, max_seconds: float,
                     max_chars: int, max_gap: float) -> list[Cue]:
    cues: list[Cue] = []
    current: list = []
    for seg in segments:
        for word in seg.words:
            if current and should_start_new_cue(current, word, max_seconds, max_chars, max_gap):
                cues.append(Cue(current[0].start, current[-1].end, words_to_text(current)))
                current = []
            current.append(word)
    if current:
        cues.append(Cue(current[0].start, current[-1].end, words_to_text(current)))
    return cues


def format_timestamp(seconds: float) -> str:
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    ms = int((seconds - int(seconds)) * 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def cues_to_srt(cues: list[Cue]) -> str:
    lines: list[str] = []
    for i, cue in enumerate(cues, start=1):
        lines.append(str(i))
        lines.append(f"{format_timestamp(cue.start)} --> {format_timestamp(cue.end)}")
        lines.append(cue.text)
        lines.append("")
    return "\n".join(lines)


def transcribe_file(pipeline, video: Path, output_dir: Path | None,
                    index: int = 0, total: int = 0,
                    language: str = "pt", beam_size: int = 5,
                    batch_size: int = 8, max_seconds: float = 6.0,
                    max_chars: int = 84, max_gap: float = 1.0) -> Path:
    dest = (output_dir or video.parent) / (video.stem + ".srt")
    batch = f" ({index}/{total})" if total else ""
    print(f"  → {video.name}{batch}")

    start = time.perf_counter()
    segments_gen, info = pipeline.transcribe(
        str(video),
        language=language,
        beam_size=beam_size,
        batch_size=batch_size,
        without_timestamps=False,
        word_timestamps=True,
        vad_filter=True,
        vad_parameters=dict(min_silence_duration_ms=500),
    )

    duration = info.duration
    collected = []
    bar = tqdm(
        total=int(duration),
        unit="s",
        bar_format="    {l_bar}{bar}| {n_fmt}/{total_fmt}s [{elapsed}<{remaining}]",
        leave=False,
    )
    prev_end = 0.0
    for seg in segments_gen:
        collected.append(seg)
        delta = seg.end - prev_end
        if delta > 0:
            bar.update(int(delta))
            prev_end = seg.end
    bar.update(bar.total - bar.n)
    bar.close()

    elapsed = time.perf_counter() - start

    if not collected:
        print(f"    ⚠ nenhuma fala detectada ({elapsed:.1f}s)")
        return dest

    collected.sort(key=lambda s: s.start)
    cues = segments_to_cues(collected, max_seconds, max_chars, max_gap)
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text(cues_to_srt(cues), encoding="utf-8")
    print(f"    ✓ {len(cues)} legendas — {elapsed:.1f}s → {dest.name}")
    return dest

## 6. Carregar o modelo

Usa `float16` na GPU (rápido e enxuto na VRAM do T4) ou `int8` como fallback de CPU. O modelo é então envolvido pelo `BatchedInferencePipeline`, que faz a inferência em paralelo.

In [ ]:
import ctranslate2
from faster_whisper import BatchedInferencePipeline, WhisperModel

if ctranslate2.get_cuda_device_count() > 0:
    device, compute_type = "cuda", "float16"
else:
    device, compute_type = "cpu", "int8"
    print("[!] GPU não detectada — rodando em CPU (vai ser lento). Troque o runtime para T4.")

model = WhisperModel(MODEL_SIZE, device=device, compute_type=compute_type)
pipeline = BatchedInferencePipeline(model=model)
print(f"[*] Modelo: {MODEL_SIZE} | Dispositivo: {device} ({compute_type}) | batch_size={BATCH_SIZE}")

## 7. Transcrever

Pula vídeos que já têm legenda (a menos que `FORCE=True`). Erros em um arquivo não interrompem o lote.

In [ ]:
videos = collect_videos(INPUT_PATH, SCAN_DEPTH)
if not videos:
    raise SystemExit(f"Nenhum vídeo encontrado em {INPUT_PATH} (SCAN_DEPTH={SCAN_DEPTH})")

input_path = Path(INPUT_PATH)
input_root = input_path if input_path.is_dir() else input_path.parent
output_root = Path(OUTPUT_PATH) if OUTPUT_PATH else None


def output_dir_for(video: Path) -> Path | None:
    if output_root is None:
        return None
    return output_root / video.parent.relative_to(input_root)


if not FORCE:
    filtered = []
    for v in videos:
        sub = existing_subtitle(v, output_dir_for(v))
        if sub:
            print(f"[=] pulando {v.name} — já tem legenda ({sub.name})")
        else:
            filtered.append(v)
    videos = filtered

if not videos:
    print("\n[✓] Nada a fazer — todos os vídeos já têm legenda")
else:
    print(f"[*] {len(videos)} vídeo(s) para transcrever\n")
    for i, video in enumerate(videos, start=1):
        try:
            transcribe_file(pipeline, video, output_dir_for(video),
                            index=i, total=len(videos),
                            language=LANGUAGE, beam_size=BEAM_SIZE,
                            batch_size=BATCH_SIZE,
                            max_seconds=MAX_SUBTITLE_SECONDS,
                            max_chars=MAX_SUBTITLE_CHARS,
                            max_gap=MAX_GAP_SECONDS)
        except Exception as e:
            print(f"    ✗ erro em {video.name}: {e}")
    print("\n[✓] Concluído")